<a href="https://colab.research.google.com/github/pritisanap2008-cyber/taxi-demand-prediction/blob/main/taxi_demand_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
import pandas as pd

df = pd.read_parquet("yellow_tripdata_2024-01.parquet",
                     columns=["tpep_pickup_datetime", "PULocationID"])

print(df.shape)
print(df.head())
print(df.dtypes)
print(df["tpep_pickup_datetime"].min(), df["tpep_pickup_datetime"].max())

(2964624, 2)
  tpep_pickup_datetime  PULocationID
0  2024-01-01 00:57:55           186
1  2024-01-01 00:03:00           140
2  2024-01-01 00:17:06           236
3  2024-01-01 00:36:38            79
4  2024-01-01 00:46:51           211
tpep_pickup_datetime    datetime64[us]
PULocationID                     int32
dtype: object
2002-12-31 22:59:39 2024-02-01 00:01:15


In [7]:
df = df.rename(columns={"tpep_pickup_datetime": "pickup_time", "PULocationID": "zone"})
print("Rows before cleaning:", len(df))

df = df[(df["pickup_time"] >= "2024-01-01") & (df["pickup_time"] < "2024-02-01")]
print("Rows after keeping January 2024 only:", len(df))

top_zones = df["zone"].value_counts().head(30).index
df = df[df["zone"].isin(top_zones)]
print("Rows in the 30 busiest zones:", len(df))

df["slot"] = df["pickup_time"].dt.floor("30min").astype("datetime64[ns]")

all_slots = pd.date_range("2024-01-01", "2024-01-31 23:30", freq="30min")
full_index = pd.MultiIndex.from_product([top_zones, all_slots], names=["zone", "slot"])

counts = df.groupby(["zone", "slot"]).size().reindex(full_index, fill_value=0)
counts = counts.reset_index(name="pickups")

print(counts.shape)
print(counts.head())
print(counts["pickups"].describe())

Rows before cleaning: 2964624
Rows after keeping January 2024 only: 2964606
Rows in the 30 busiest zones: 2419051
(44640, 3)
   zone                slot  pickups
0   132 2024-01-01 00:00:00       63
1   132 2024-01-01 00:30:00       87
2   132 2024-01-01 01:00:00       46
3   132 2024-01-01 01:30:00       22
4   132 2024-01-01 02:00:00       12
count    44640.000000
mean        54.190211
std         48.921208
min          0.000000
25%         14.000000
50%         45.000000
75%         78.000000
max        435.000000
Name: pickups, dtype: float64


In [8]:
counts = counts.sort_values(["zone", "slot"]).reset_index(drop=True)

counts["slot_of_day"] = counts["slot"].dt.hour * 2 + counts["slot"].dt.minute // 30
counts["day_of_week"] = counts["slot"].dt.dayofweek
counts["is_weekend"] = (counts["day_of_week"] >= 5).astype(int)

grouped = counts.groupby("zone")["pickups"]
counts["lag_1"] = grouped.shift(1)
counts["lag_2"] = grouped.shift(2)
counts["lag_3"] = grouped.shift(3)
counts["lag_48"] = grouped.shift(48)
counts["avg_last_4"] = grouped.transform(lambda s: s.shift(1).rolling(4).mean())

counts = counts.dropna().reset_index(drop=True)

print(counts.shape)
print(counts.head())

(43200, 11)
   zone                slot  pickups  slot_of_day  day_of_week  is_weekend  \
0    43 2024-01-02 00:00:00        2            0            1           0   
1    43 2024-01-02 00:30:00        1            1            1           0   
2    43 2024-01-02 01:00:00        0            2            1           0   
3    43 2024-01-02 01:30:00        0            3            1           0   
4    43 2024-01-02 02:00:00        0            4            1           0   

   lag_1  lag_2  lag_3  lag_48  avg_last_4  
0    3.0    2.0    4.0    90.0        4.50  
1    2.0    3.0    2.0    72.0        2.75  
2    1.0    2.0    3.0    49.0        2.00  
3    0.0    1.0    2.0    40.0        1.50  
4    0.0    0.0    1.0    29.0        0.75  


In [9]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

train = counts[counts["slot"] < "2024-01-25"].copy()
test = counts[counts["slot"] >= "2024-01-25"].copy()

zone_avg = train.groupby("zone")["pickups"].mean()
train["zone_avg"] = train["zone"].map(zone_avg)
test["zone_avg"] = test["zone"].map(zone_avg)

features = ["slot_of_day", "day_of_week", "is_weekend", "lag_1", "lag_2",
            "lag_3", "lag_48", "avg_last_4", "zone_avg"]
X_train, y_train = train[features], train["pickups"]
X_test, y_test = test[features], test["pickups"]
print(X_train.shape, X_test.shape)

lr = LinearRegression()
lr.fit(X_train, y_train)

xgb = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1)
xgb.fit(X_train, y_train)

preds = {
    "Baseline: same time yesterday": X_test["lag_48"].values,
    "Baseline: previous slot": X_test["lag_1"].values,
    "Linear Regression": np.clip(lr.predict(X_test), 0, None),
    "XGBoost": np.clip(xgb.predict(X_test), 0, None),
}

y_arr = y_test.values
mask = y_arr >= 10

for name, p in preds.items():
    mae = mean_absolute_error(y_arr, p)
    mape = (np.abs(y_arr[mask] - p[mask]) / y_arr[mask]).mean() * 100
    print(name, "| MAE:", round(mae, 2), "| MAPE:", round(mape, 1), "%")

(33120, 9) (10080, 9)
Baseline: same time yesterday | MAE: 17.64 | MAPE: 40.6 %
Baseline: previous slot | MAE: 12.0 | MAPE: 25.8 %
Linear Regression | MAE: 11.01 | MAPE: 23.5 %
XGBoost | MAE: 9.21 | MAPE: 19.1 %
